<a href="https://colab.research.google.com/github/legna7816/ml-projects/blob/main/nlp_sentiment/nlp_step01_movie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 토큰화 - 문장을 단어 단위로 쪼개기

sentence = "I really loved this movie"
tokens = sentence.lower().split()
print(tokens)

['i', 'really', 'loved', 'this', 'movie']


In [ ]:
# 2. BoW(Bag of Words) - 단어 등장 횟수로 벡터 만들기

from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    "I loved this movie",
    "I hated this movie",
    "this movie was great",
]

vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(corpus)

print(vectorizer.get_feature_names_out())
print(X_bow.toarray())

['great' 'hated' 'loved' 'movie' 'this' 'was']
[[0 0 1 1 1 0]
 [0 1 0 1 1 0]
 [1 0 0 1 1 1]]


In [ ]:
# 3. TF-IDF

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus)

print(tfidf.get_feature_names_out())
print(X_tfidf.toarray().round(2))

['great' 'hated' 'loved' 'movie' 'this' 'was']
[[0.   0.   0.77 0.45 0.45 0.  ]
 [0.   0.77 0.   0.45 0.45 0.  ]
 [0.61 0.   0.   0.36 0.36 0.61]]


In [ ]:
# 4. TODO
# 4-1. 새 corpus 만든 후 CountVectorzier 적용해 단어 목록 출력
new_corpus = [
    "the food was delicious",
    "the food was terrible",
    "i will visit again"
]

new_vectorizer = CountVectorizer()
new_X_bow = new_vectorizer.fit_transform(new_corpus)

print(new_vectorizer.get_feature_names_out())
print(new_X_bow.toarray())

# 4-2. new_corpus로 TfidfVectorzier 적용
new_tfidf = TfidfVectorizer()
new_X_tfidf = new_tfidf.fit_transform(new_corpus)

print(new_tfidf.get_feature_names_out())
print(new_X_tfidf.toarray().round(2))

# 4-3. tfidf에 stop_words='english' 옵션 부여 후 관사가 사라졌는지 확인
new_tfidf = TfidfVectorizer(stop_words='english')
new_X_tfidf = new_tfidf.fit_transform(new_corpus)

print(new_tfidf.get_feature_names_out())
print(new_X_tfidf.toarray().round(2))

['again' 'delicious' 'food' 'terrible' 'the' 'visit' 'was' 'will']
[[0 1 1 0 1 0 1 0]
 [0 0 1 1 1 0 1 0]
 [1 0 0 0 0 1 0 1]]
['again' 'delicious' 'food' 'terrible' 'the' 'visit' 'was' 'will']
[[0.   0.6  0.46 0.   0.46 0.   0.46 0.  ]
 [0.   0.   0.46 0.6  0.46 0.   0.46 0.  ]
 [0.58 0.   0.   0.   0.   0.58 0.   0.58]]
['delicious' 'food' 'terrible' 'visit']
[[0.8  0.61 0.   0.  ]
 [0.   0.61 0.8  0.  ]
 [0.   0.   0.   1.  ]]


In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_files
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. 데이터 불러오기
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

from sklearn.datasets import fetch_20newsgroups

In [ ]:
# 더 간단한 내정 데이터: 영화 리뷰 느낌의 실습용 데이터
reviews = [
    # 긍정 리뷰 (label=1)
    "this movie was absolutely fantastic and amazing",
    "great film loved every moment of it",
    "brilliant acting and wonderful story",
    "best movie i have ever seen highly recommend",
    "incredible cinematography and beautiful music",
    "outstanding performance by all the actors",
    "a masterpiece of modern cinema loved it",
    "funny heartwarming and deeply moving film",
    "superb direction and excellent screenplay",
    "one of the greatest movies of all time",

    # 부정 리뷰 (label=0)
    "this movie was terrible waste of time",
    "awful boring and completely unwatchable",
    "worst film i have ever seen absolutely horrible",
    "bad acting poor story and dull direction",
    "completely disappointing and frustrating experience",
    "nothing works in this dreadful movie",
    "tedious and painfully boring throughout",
    "ridiculous plot with terrible performances",
    "an absolute disaster avoid at all costs",
    "poorly made and utterly forgettable film"
]
labels = [1]*10 + [0]*10

In [ ]:
# 2. 전처리 - TF-IDF 로 텍스트 벡터화
# stop_words='english' = the, is, a 같은 의미없는 단어 자동 제거
# max_feature=500: 가장 중요한 단어 500개만 사용 (큰 데이터에서 차원 제한)

tfidf = TfidfVectorizer(stop_words='english', max_features=500)
X = tfidf.fit_transform(reviews)
y = np.array(labels)

print('벡터 크기:', X.shape)
print('사용된 단어 목록:', tfidf.get_feature_names_out())

벡터 크기: (20, 65)
사용된 단어 목록: ['absolute' 'absolutely' 'acting' 'actors' 'amazing' 'avoid' 'awful' 'bad'
 'beautiful' 'best' 'boring' 'brilliant' 'cinema' 'cinematography'
 'completely' 'costs' 'deeply' 'direction' 'disappointing' 'disaster'
 'dreadful' 'dull' 'excellent' 'experience' 'fantastic' 'film'
 'forgettable' 'frustrating' 'funny' 'great' 'greatest' 'heartwarming'
 'highly' 'horrible' 'incredible' 'loved' 'masterpiece' 'modern' 'moment'
 'movie' 'movies' 'moving' 'music' 'outstanding' 'painfully' 'performance'
 'performances' 'plot' 'poor' 'poorly' 'recommend' 'ridiculous'
 'screenplay' 'seen' 'story' 'superb' 'tedious' 'terrible' 'time'
 'unwatchable' 'utterly' 'waste' 'wonderful' 'works' 'worst']


In [ ]:
# 3. 모델 학습

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 모델1: LogisticRegression (타이타닉에서도 쓴 모델)
log_model = LogisticRegression()
log_model.fit(X_train, y_train)
log_pred = log_model.predict(X_test)
print('Logistic 정확도: ', accuracy_score(y_test, log_pred))

# 모델2: MultinomialNB (나이브베이즈) - NLP에서 자주 쓰는 모델
# 이 단어들이 나왔을 때, 긍정일 확률과 부정일 확률을 계산하는 방식
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)
print('NaiveBayes 정확도', accuracy_score(y_test, nb_pred))
print(classification_report(y_test, nb_pred, target_names=['부정', '긍정']))

Logistic 정확도:  0.3333333333333333
NaiveBayes 정확도 0.3333333333333333
              precision    recall  f1-score   support

          부정       0.33      1.00      0.50         2
          긍정       0.00      0.00      0.00         4

    accuracy                           0.33         6
   macro avg       0.17      0.50      0.25         6
weighted avg       0.11      0.33      0.17         6



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# 4. 새로운 리뷰 직접 예측

new_reviews = [
    "this film was absolutely wonderful and fantastic",
    "terrible movie boring and awful acting"
]
X_new = tfidf.transform(new_reviews)
print('예측결과 (1=긍정, 0=부정):', log_model.predict(X_new))

예측결과 (1=긍정, 0=부정): [0 0]


In [ ]:
# 5. TODO
# 5-1. 교차검증
log_cv = cross_val_score(LogisticRegression(max_iter=200),X, y, cv=5)
nb_cv = cross_val_score(MultinomialNB(), X, y, cv=5)

print('Logistic 평균:', log_cv.mean())
print('NaiveBayes 평균:', nb_cv.mean())

# 5-2. 새로운 리뷰2를 생성 및 비교
new_reviews2 = ["incredible story with great acting and beautiful direciton"]
X_new2 = tfidf.transform(new_reviews2)
print('예측결과:', log_model.predict(X_new2))

# 5-3. tfidf에서 fit_transform 대신 transform만 써야 하는 이유
# tfidf는 리뷰 텍스트를 벡터화하기 위해 사용함

Logistic 평균: 0.45
NaiveBayes 평균: 0.5
예측결과: [1]
